# Airline Customer Feedback Knowledge Graph Using Neo4j and Cypher

## Project Description

This project builds a knowledge graph from airline customer feedback collected from Twitter. The workflow includes data preparation, ontology  definition, semantic modeling, Neo4j graph construction, and Cypher-based graph analysis.

The project models airlines, customer feedback, sentiment, and complaint categories as connected entities. The resulting Neo4j knowledge graph enables relationship-based analysis of customer sentiment and complaints and supports retrieval of specific customer feedback through Cypher queries.  

In [1]:
# !pip install neo4j

## Import libraries

In [2]:
import pandas as pd
import numpy as np

from neo4j import GraphDatabase

## Load dataset

In [3]:
df = pd.read_csv(r"D:\ds ai ml\DS@AI\00-portfolio-projects/Tweets.csv")

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

df.head()

Rows: 14640
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
# Select Required Features for Knowledge Graph Construction
required_columns = [
    "tweet_id",
    "airline",
    "airline_sentiment",
    "airline_sentiment_confidence",
    "negativereason",
    "negativereason_confidence",
    "text",
    "tweet_created"
]

df[required_columns].head()

,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,text,tweet_created
0,570306133677760513,Virgin America,neutral,1.0000,NaN,NaN,@VirginAmerica What @dhepburn said.,2015-02-24 11:35:52 -0800
1,570301130888122368,Virgin America,positive,0.3486,NaN,0.0000,@VirginAmerica plus you've added commercials t...,2015-02-24 11:15:59 -0800
2,570301083672813571,Virgin America,neutral,0.6837,NaN,NaN,@VirginAmerica I didn't today... Must mean I n...,2015-02-24 11:15:48 -0800
3,570301031407624196,Virgin America,negative,1.0000,Bad Flight,0.7033,@VirginAmerica it's really aggressive to blast...,2015-02-24 11:15:36 -0800
4,570300817074462722,Virgin America,negative,1.0000,Can't Tell,1.0000,@VirginAmerica and it's a really big bad thing...,2015-02-24 11:14:45 -0800


In [5]:
# Prepare the data
kg_df = df[required_columns].copy()

In [6]:
# Handle missing values
kg_df["text"] = kg_df["text"].fillna("").astype(str)

kg_df["airline"] = (
    kg_df["airline"]
    .fillna("Unknown")
    .astype(str)
)

kg_df["airline_sentiment"] = (
    kg_df["airline_sentiment"]
    .fillna("unknown")
    .astype(str)
    .str.lower()
)

kg_df["negativereason"] = (
    kg_df["negativereason"]
    .fillna("Not Applicable")
    .astype(str)
)

In [7]:
kg_df.head()

,tweet_id,airline,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,text,tweet_created
0,570306133677760513,Virgin America,neutral,1.0000,Not Applicable,NaN,@VirginAmerica What @dhepburn said.,2015-02-24 11:35:52 -0800
1,570301130888122368,Virgin America,positive,0.3486,Not Applicable,0.0000,@VirginAmerica plus you've added commercials t...,2015-02-24 11:15:59 -0800
2,570301083672813571,Virgin America,neutral,0.6837,Not Applicable,NaN,@VirginAmerica I didn't today... Must mean I n...,2015-02-24 11:15:48 -0800
3,570301031407624196,Virgin America,negative,1.0000,Bad Flight,0.7033,@VirginAmerica it's really aggressive to blast...,2015-02-24 11:15:36 -0800
4,570300817074462722,Virgin America,negative,1.0000,Can't Tell,1.0000,@VirginAmerica and it's a really big bad thing...,2015-02-24 11:14:45 -0800


In [8]:
kg_df = kg_df[:100]

## Ontology

The project uses a lightweight domain ontology.

Entities:

1. Airline
2. Feedback
3. Sentiment
4. Complaint

Relationships:

Airline → HAS_FEEDBACK → Feedback

Feedback → HAS_SENTIMENT → Sentiment

Feedback → HAS_COMPLAINT → Complaint

The ontology defines the important concepts and the allowed
relationships between them.

In [9]:
ontology = {
    "entities": {
        "Airline": "An airline receiving customer feedback",
        "Feedback": "A customer feedback record",
        "Sentiment": "The sentiment associated with feedback",
        "Complaint": "A complaint category associated with feedback"
    },

    "relationships": {
        "HAS_FEEDBACK": "Connects an airline to customer feedback",
        "HAS_SENTIMENT": "Connects feedback to its sentiment",
        "HAS_COMPLAINT": "Connects feedback to its complaint category"
    }
}

ontology

{'entities': {'Airline': 'An airline receiving customer feedback',
  'Feedback': 'A customer feedback record',
  'Sentiment': 'The sentiment associated with feedback',
  'Complaint': 'A complaint category associated with feedback'},
 'relationships': {'HAS_FEEDBACK': 'Connects an airline to customer feedback',
  'HAS_SENTIMENT': 'Connects feedback to its sentiment',
  'HAS_COMPLAINT': 'Connects feedback to its complaint category'}}

In [10]:
print("Entities:")

for entity, description in ontology["entities"].items():
    print(f"- {entity}: {description}")

print("\nRelationships:")

for relationship, description in ontology["relationships"].items():
    print(f"- {relationship}: {description}")

Entities:
- Airline: An airline receiving customer feedback
- Feedback: A customer feedback record
- Sentiment: The sentiment associated with feedback
- Complaint: A complaint category associated with feedback

Relationships:
- HAS_FEEDBACK: Connects an airline to customer feedback
- HAS_SENTIMENT: Connects feedback to its sentiment
- HAS_COMPLAINT: Connects feedback to its complaint category


## Semantic Modeling

In [11]:
semantic_model = {
    "Airline": {
        "properties": ["name"]
    },

    "Feedback": {
        "properties": [
            "tweet_id",
            "text",
            "sentiment_confidence",
            "complaint_confidence",
            "created_at"
        ]
    },

    "Sentiment": {
        "properties": ["name"]
    },

    "Complaint": {
        "properties": ["name"]
    }
}

semantic_model

{'Airline': {'properties': ['name']},
 'Feedback': {'properties': ['tweet_id',
   'text',
   'sentiment_confidence',
   'complaint_confidence',
   'created_at']},
 'Sentiment': {'properties': ['name']},
 'Complaint': {'properties': ['name']}}

## Connect to Neo4j

In [12]:
# Understand the graph before creating it
# Connect to Neo4j
NEO4J_URI = "neo4j+s://a320ebe7.databases.neo4j.io"
NEO4J_USERNAME = "a320ebe7"
NEO4J_PASSWORD = "KwsVLXc93iEW62DYXWcls2FlNyndW7Fcp_LzZwf3AO4"

In [13]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Successfully connected to Neo4j")

Successfully connected to Neo4j


## Create Graph Constraints

In [14]:
constraints = [
    """
    CREATE CONSTRAINT airline_name_unique IF NOT EXISTS
    FOR (a:Airline)
    REQUIRE a.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT sentiment_name_unique IF NOT EXISTS
    FOR (s:Sentiment)
    REQUIRE s.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT complaint_name_unique IF NOT EXISTS
    FOR (c:Complaint)
    REQUIRE c.name IS UNIQUE
    """,

    """
    CREATE CONSTRAINT feedback_id_unique IF NOT EXISTS
    FOR (f:Feedback)
    REQUIRE f.tweet_id IS UNIQUE
    """
]

with driver.session() as session:

    for query in constraints:
        session.run(query)

print("Constraints created.")

Constraints created.


In [15]:
# Clear old graph
# with driver.session() as session:
#     session.run("MATCH (n) DETACH DELETE n")

# print("Existing graph cleared.")

## Construct the Knowledge Graph Using Cypher

In [16]:
# Create Knowledge Graph using Cypher
create_feedback_query = """
MERGE (a:Airline {
    name: $airline
})

MERGE (s:Sentiment {
    name: $sentiment
})

MERGE (c:Complaint {
    name: $complaint
})

MERGE (f:Feedback {
    tweet_id: $tweet_id
})

SET
    f.text = $text,
    f.sentiment_confidence = $sentiment_confidence,
    f.complaint_confidence = $complaint_confidence,
    f.created_at = $created_at

MERGE (a)-[:HAS_FEEDBACK]->(f)

MERGE (f)-[:HAS_SENTIMENT]->(s)

MERGE (f)-[:HAS_COMPLAINT]->(c)
"""

In [17]:
# insert tweets
with driver.session() as session:

    for _, row in kg_df.iterrows():

        session.run(
            create_feedback_query,

            airline=row["airline"],

            tweet_id=str(row["tweet_id"]),

            text=row["text"],

            sentiment=row["airline_sentiment"],

            complaint=row["negativereason"],

            sentiment_confidence=(
                float(row["airline_sentiment_confidence"])
                if pd.notna(row["airline_sentiment_confidence"])
                else None
            ),

            complaint_confidence=(
                float(row["negativereason_confidence"])
                if pd.notna(row["negativereason_confidence"])
                else None
            ),

            created_at=(
                str(row["tweet_created"])
                if pd.notna(row["tweet_created"])
                else None
            )
        )

print("Knowledge Graph created successfully.")

Knowledge Graph created successfully.


In [18]:
# Check number of nodes
query = """
MATCH (n)
RETURN labels(n) AS node_type, count(n) AS count
ORDER BY count DESC
"""

with driver.session() as session:

    result = session.run(query)

    for record in result:
        print(record["node_type"], record["count"])

['Feedback'] 758
['Complaint'] 11
['Sentiment'] 3
['Airline'] 2


## Cypher Query Function

In [19]:
def run_cypher(query, parameters=None):
    with driver.session() as session:
        result = session.run(query, parameters or {})
        return list(result)

In [20]:
## Verify Node Distribution in the Knowledge Graph
result = run_cypher("""
MATCH (n)
RETURN labels(n) AS labels, count(n) AS count
ORDER BY count DESC
""")

for record in result:
    print(record)

<Record labels=['Feedback'] count=758>
<Record labels=['Complaint'] count=11>
<Record labels=['Sentiment'] count=3>
<Record labels=['Airline'] count=2>


In [21]:
# Cypher Query 1 — List airlines
result = run_cypher("""
MATCH (a:Airline)
RETURN a.name AS airline
ORDER BY airline
""")

for record in result:
    print(record["airline"])

United
Virgin America


In [22]:
# Cypher Query 2 — Count feedback by airline
result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
RETURN
    a.name AS airline,
    count(f) AS feedback_count
ORDER BY feedback_count DESC
""")

for record in result:
    print(record["airline"], record["feedback_count"])

Virgin America 504
United 254


In [24]:
# Cypher Query 3 — Sentiment distribution
result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
      -[:HAS_SENTIMENT]->(s:Sentiment)
RETURN
    a.name AS airline,
    s.name AS sentiment,
    count(f) AS feedback_count
ORDER BY airline, feedback_count DESC
""")

for record in result:
    print(
        record["airline"],
        record["sentiment"],
        record["feedback_count"]
    )

United negative 168
United neutral 56
United positive 30
Virgin America negative 181
Virgin America neutral 171
Virgin America positive 152


In [25]:
# Cypher Query 4 — Most common complaints

result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
      -[:HAS_SENTIMENT]->(s:Sentiment)
RETURN
    a.name AS airline,
    s.name AS sentiment,
    count(f) AS feedback_count
ORDER BY airline, feedback_count DESC
""")

for record in result:
    print(
        record["airline"],
        record["sentiment"],
        record["feedback_count"]
    )

United negative 168
United neutral 56
United positive 30
Virgin America negative 181
Virgin America neutral 171
Virgin America positive 152


In [26]:
# Cypher Query 5 — Negative complaints by airline

result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
      -[:HAS_SENTIMENT]->(s:Sentiment),
      (f)-[:HAS_COMPLAINT]->(c:Complaint)
WHERE s.name = "negative"
  AND c.name <> "Not Applicable"
RETURN
    a.name AS airline,
    c.name AS complaint,
    count(f) AS complaint_count
ORDER BY complaint_count DESC
""")

for record in result:
    print(
        record["airline"],
        record["complaint"],
        record["complaint_count"]
    )

Virgin America Customer Service Issue 60
United Customer Service Issue 57
Virgin America Flight Booking Problems 28
United Can't Tell 28
Virgin America Can't Tell 22
United Lost Luggage 21
United Late Flight 20
Virgin America Bad Flight 19
Virgin America Cancelled Flight 18
Virgin America Late Flight 17
United Flight Attendant Complaints 11
United Bad Flight 9
United Flight Booking Problems 9
United Cancelled Flight 8
Virgin America Lost Luggage 5
Virgin America Flight Attendant Complaints 5
Virgin America Damaged Luggage 4
Virgin America longlines 3
United Damaged Luggage 3
United longlines 2


In [27]:
# Cypher Query 6 — Investigate one airline

result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
      -[:HAS_SENTIMENT]->(s:Sentiment),
      (f)-[:HAS_COMPLAINT]->(c:Complaint)
WHERE a.name = "United"
  AND s.name = "negative"
  AND c.name <> "Not Applicable"
RETURN
    c.name AS complaint,
    count(f) AS complaint_count
ORDER BY complaint_count DESC
""")

for record in result:
    print(
        record["complaint"],
        record["complaint_count"]
    )

Customer Service Issue 57
Can't Tell 28
Lost Luggage 21
Late Flight 20
Flight Attendant Complaints 11
Bad Flight 9
Flight Booking Problems 9
Cancelled Flight 8
Damaged Luggage 3
longlines 2


In [28]:
# Cypher Query 7 — Retrieve actual customer feedback

result = run_cypher("""
MATCH (a:Airline)-[:HAS_FEEDBACK]->(f:Feedback)
      -[:HAS_COMPLAINT]->(c:Complaint)
WHERE a.name = "United"
  AND c.name = "Late Flight"
RETURN
    f.tweet_id AS tweet_id,
    f.text AS customer_feedback
LIMIT 10
""")

for record in result:
    print(
        record["tweet_id"],
        ":", 
        record["customer_feedback"]
    )

570307026263384064 : @united Delayed due to lack of crew and now delayed again because there's a long line for deicing... Still need to improve service #united
570289777184002048 : @united See? We were told repeatedly that the pilot was Late Flight and kept getting Late Flightr.  After we boarded, there was a defibrillator issue.
570280548922499073 : @United well sitting on the ground 'on time' but waiting for a gate....again #tiredofthis
570277667519332353 : @united A measly $50 e-certificate is not how you appreciate loyal customers after they wait 3hrs on the tarmac during UA1116. #unacceptable
570271644473622528 : @united in addition, my first flight was delayed an hour and I'm arriving at my destination 8 hrs Late Flight.
570254368538173440 : @united I tried but no one was available in bogota and everyone was rude in Houston. I was stuck for 35 hours because of you guys
570252666439385088 : @United. What's going on with UA 236?  outbound flight last thurs was delayed 4hrs How long

## Conclusion

-  This project demonstrates the construction and analysis of an airline customer-feedback knowledge graph using Python, Neo4j, and Cypher. The workflow begins with data preparation and progresses through ontology definition, semantic modeling, graph construction, and relationship-based querying.

-  The ontology models Airline, Feedback, Sentiment, and Complaint as connected entities, while Neo4j stores these entities and their semantic relationships as a property graph. Cypher queries are then used to traverse the graph, analyze sentiment and complaint patterns, and retrieve context-specific customer feedback.

-  The project demonstrates how a knowledge graph can transform tabular customer-feedback data into a connected representation that supports relationship-oriented analysis and business insight generation